In [41]:
import json
import time
import os

import pandas as pd
import numpy as np
from openai import OpenAI
from google import genai
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

EVALUATOR_MODEL = "gpt-5.4-mini"

print("OpenAI client ready.")

OpenAI client ready.


**Eval 4 — Temporal consistency**

In [7]:
with open("rag_summaries.json", "r") as f:
    rag_summaries = json.load(f)

print("Loaded summaries:", len(rag_summaries))
print(rag_summaries.keys())

Loaded summaries: 9
dict_keys(['Whole + Gemini', 'Whole + BGE', 'Whole + MedCPT', 'Fixed + Gemini', 'Fixed + BGE', 'Fixed + MedCPT', 'Section + Gemini', 'Section + BGE', 'Section + MedCPT'])


In [8]:
temporal_events_df = pd.read_csv(
    "temporal_candidate_events.csv"
)

In [9]:
TEMPORAL_CONSOLIDATION_PROMPT = """
You are consolidating clinical events for longitudinal temporal evaluation.

The source clinical notes have already been ordered by creation_timestamp.
The event's source note index therefore represents its authoritative
chronological position.

For each CURRENT candidate event, determine whether it represents:

KEEP:
- a genuinely new clinical event
- a new investigation, treatment, diagnosis, referral, or disposition
- a meaningful change in clinical state
- a new value/finding that represents clinical progression
- a treatment being started, stopped, changed, or meaningfully continued

REMOVE:
- the same clinical event already represented in EARLIER events
- historical information merely restated
- duplicated findings/results with no new clinical change
- administrative/non-clinically meaningful information

Important:
Do NOT remove genuine progression simply because it concerns the same
clinical concept.

Example:
"SpO2 89% on room air"
"SpO2 improved to 92% after oxygen"
"SpO2 later 94% on room air"
are separate meaningful states and should all be kept.

But repeated statements of the same NT-proBNP result of 4500 pg/mL,
without a new measurement or change, should not create multiple events.

Do not use dates written inside event text to establish chronology.
Use the supplied source note indices only.

Return valid JSON only:

{
  "decisions": [
    {
      "candidate_id": 0,
      "decision": "KEEP" or "REMOVE",
      "reason": "brief reason"
    }
  ]
}
"""

In [10]:
def consolidate_note_events(current_events, earlier_events):

    # Give each current candidate an ID so we can map
    # Nemotron's decision back to the original event.
    current_candidates = [
        {
            "candidate_id": i,
            "event": event
        }
        for i, event in enumerate(current_events)
    ]

    user_message = f"""
EARLIER CONSOLIDATED EVENTS:
{json.dumps(earlier_events, indent=2)}

CURRENT CANDIDATE EVENTS:
{json.dumps(current_candidates, indent=2)}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": TEMPORAL_CONSOLIDATION_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    # Get the model output safely.
    raw_output = response.choices[0].message.content

    if raw_output is None or not raw_output.strip():
        raise ValueError(
            "Nemotron returned an empty response."
        )

    # Remove Markdown fences if the model returns ```json ... ```
    raw_output = raw_output.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.replace("```json", "")
        raw_output = raw_output.replace("```", "")
        raw_output = raw_output.strip()

    result = json.loads(raw_output)

    return result["decisions"]

In [11]:
test_consolidated_events = []
test_decisions = []

for note_index in [0, 1, 2]:

    current_df = temporal_events_df[
        temporal_events_df["source_note_index"] == note_index
    ]

    current_events = current_df["event"].tolist()

    decisions = consolidate_note_events(
        current_events=current_events,
        earlier_events=[
            item["event"]
            for item in test_consolidated_events
        ]
    )

    for decision in decisions:

        candidate_id = decision["candidate_id"]
        event_text = current_events[candidate_id]

        record = {
            "source_note_index": note_index,
            "event": event_text,
            "decision": decision["decision"],
            "reason": decision["reason"]
        }

        test_decisions.append(record)

        if decision["decision"] == "KEEP":
            test_consolidated_events.append({
                "source_note_index": note_index,
                "event": event_text
            })


for item in test_decisions:
    print(
        f"NOTE {item['source_note_index']} | "
        f"{item['decision']} | "
        f"{item['event']}"
    )
    print("   Reason:", item["reason"])

NOTE 0 | KEEP | Patient presented with progressive breathlessness
   Reason: New presenting symptom indicating current clinical state
NOTE 0 | KEEP | Triage category 2 assigned for high-risk, potentially life-threatening condition
   Reason: New triage assessment reflecting acuity
NOTE 0 | KEEP | ED diagnosis of chronic thromboembolic pulmonary hypertension established
   Reason: New diagnosis established in ED
NOTE 0 | KEEP | Oxygen saturation recorded at 89% on room air
   Reason: New objective hypoxemia finding
NOTE 0 | KEEP | Low-flow oxygen therapy initiated
   Reason: New treatment started
NOTE 0 | KEEP | Admission to Respiratory High Dependency Unit planned
   Reason: New disposition/admission plan
NOTE 1 | REMOVE | Patient presented with progressive breathlessness
   Reason: Same presenting symptom already captured earlier.
NOTE 1 | REMOVE | Diagnosed with chronic thromboembolic pulmonary hypertension
   Reason: Duplicate of the established chronic thromboembolic pulmonary hype

In [12]:
consolidated_events = []
consolidation_decisions = []

for note_index in sorted(
    temporal_events_df["source_note_index"].unique()
):

    current_df = temporal_events_df[
        temporal_events_df["source_note_index"] == note_index
    ]

    current_events = current_df["event"].tolist()

    decisions = consolidate_note_events(
        current_events=current_events,
        earlier_events=[
            item["event"]
            for item in consolidated_events
        ]
    )

    kept_count = 0

    for decision in decisions:

        candidate_id = decision["candidate_id"]
        event_text = current_events[candidate_id]

        decision_record = {
            "source_note_index": note_index,
            "event": event_text,
            "decision": decision["decision"],
            "reason": decision["reason"]
        }

        consolidation_decisions.append(decision_record)

        if decision["decision"] == "KEEP":
            consolidated_events.append({
                "source_note_index": note_index,
                "event": event_text
            })
            kept_count += 1

    print(
        f"NOTE {note_index:02d} | "
        f"{len(current_events)} candidates | "
        f"{kept_count} kept"
    )

    # Save progress after every note.
    pd.DataFrame(consolidated_events).to_csv(
        "temporal_consolidated_events_gpt54mini.csv",
        index=False
    )

    pd.DataFrame(consolidation_decisions).to_csv(
        "temporal_consolidation_decisions_gpt54mini.csv",
        index=False
    )

print("\nDONE")
print("Final consolidated events:", len(consolidated_events))

NOTE 00 | 6 candidates | 6 kept
NOTE 01 | 6 candidates | 2 kept
NOTE 02 | 4 candidates | 1 kept
NOTE 03 | 7 candidates | 4 kept
NOTE 04 | 12 candidates | 8 kept
NOTE 05 | 12 candidates | 4 kept
NOTE 06 | 7 candidates | 2 kept
NOTE 07 | 4 candidates | 3 kept
NOTE 08 | 2 candidates | 1 kept
NOTE 09 | 8 candidates | 7 kept
NOTE 10 | 6 candidates | 0 kept
NOTE 11 | 3 candidates | 3 kept
NOTE 12 | 9 candidates | 6 kept
NOTE 13 | 4 candidates | 3 kept
NOTE 14 | 4 candidates | 4 kept
NOTE 15 | 5 candidates | 1 kept
NOTE 16 | 8 candidates | 2 kept
NOTE 17 | 6 candidates | 3 kept
NOTE 18 | 9 candidates | 3 kept
NOTE 19 | 6 candidates | 2 kept
NOTE 20 | 1 candidates | 1 kept
NOTE 21 | 7 candidates | 5 kept
NOTE 22 | 6 candidates | 0 kept
NOTE 23 | 4 candidates | 1 kept
NOTE 24 | 5 candidates | 4 kept
NOTE 25 | 7 candidates | 4 kept
NOTE 26 | 6 candidates | 1 kept
NOTE 27 | 8 candidates | 4 kept
NOTE 28 | 5 candidates | 1 kept
NOTE 29 | 2 candidates | 1 kept
NOTE 30 | 5 candidates | 2 kept
NOTE 3

In [13]:
SUMMARY_EVENT_MAPPING_PROMPT = """
You are aligning a generated longitudinal clinical summary with a
reference timeline of clinical events.

The reference events are already in authoritative chronological order,
based on the source clinical notes' creation_timestamp.

For each reference event, determine whether the underlying clinical event
is represented in the generated summary.

MATCH:
- The summary clearly expresses the same underlying clinical event.
- Paraphrasing is allowed.
- Exact wording is not required.
- A summary sentence may represent more than one reference event.

NO_MATCH:
- The event is absent from the summary.
- The summary only contains a vague related statement that does not
  establish the reference event.
- The clinical content is materially different.

Important:
- Do NOT judge whether the summary event occurs in the correct chronological
  position. Your task is only to identify correspondence and where the
  matched event appears in the summary.
- Do NOT penalize omissions.
- Do NOT judge hallucinations or unsupported claims.
- Do NOT use outside medical knowledge.
- Do NOT use dates inside the reference events to change their reference
  chronology.
- Match only on the clinical content of the event.

The generated summary is divided into numbered units in the order they
appear. For every matched reference event, return the number of the
summary unit that represents it.

Return valid JSON only:

{
  "matches": [
    {
      "reference_event_id": 0,
      "match": "MATCH",
      "summary_unit": 3
    },
    {
      "reference_event_id": 1,
      "match": "NO_MATCH",
      "summary_unit": null
    }
  ]
}
"""

In [14]:
import re


def split_summary_into_units(summary):
    """
    Split a generated summary into ordered text units.

    Each unit keeps its original position in the summary so that
    we can later measure the order of matched clinical events.
    """

    # Normalize whitespace while preserving the text itself.
    text = re.sub(r"\s+", " ", summary.strip())

    # Split primarily at sentence boundaries.
    sentences = re.split(r"(?<=[.!?])\s+", text)

    # Remove empty pieces.
    sentences = [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

    # Give every unit an order number.
    units = [
        {
            "summary_unit": i,
            "text": sentence
        }
        for i, sentence in enumerate(sentences, start=1)
    ]

    return units

In [15]:
summary_units = {
    config: split_summary_into_units(summary)
    for config, summary in rag_summaries.items()
}

for config, units in summary_units.items():
    print(f"{config}: {len(units)} units")

Whole + Gemini: 43 units
Whole + BGE: 33 units
Whole + MedCPT: 28 units
Fixed + Gemini: 35 units
Fixed + BGE: 33 units
Fixed + MedCPT: 36 units
Section + Gemini: 35 units
Section + BGE: 25 units
Section + MedCPT: 52 units


In [16]:
reference_timeline = [
    {
        "reference_event_id": i,
        "source_note_index": event["source_note_index"],
        "event": event["event"]
    }
    for i, event in enumerate(consolidated_events)
]

print("Reference events:", len(reference_timeline))

# Quick look at the beginning
for event in reference_timeline[:5]:
    print(event)

Reference events: 116
{'reference_event_id': 0, 'source_note_index': np.int64(0), 'event': 'Patient presented with progressive breathlessness'}
{'reference_event_id': 1, 'source_note_index': np.int64(0), 'event': 'Triage category 2 assigned for high-risk, potentially life-threatening condition'}
{'reference_event_id': 2, 'source_note_index': np.int64(0), 'event': 'ED diagnosis of chronic thromboembolic pulmonary hypertension established'}
{'reference_event_id': 3, 'source_note_index': np.int64(0), 'event': 'Oxygen saturation recorded at 89% on room air'}
{'reference_event_id': 4, 'source_note_index': np.int64(0), 'event': 'Low-flow oxygen therapy initiated'}


In [18]:
def map_reference_events_to_summary(reference_timeline, units):

    # Send numbered summary units so the model only has to
    # identify correspondence and location.
    summary_for_model = [
        {
            "summary_unit": unit["summary_unit"],
            "text": unit["text"]
        }
        for unit in units
    ]

    user_message = f"""
REFERENCE TIMELINE:
{json.dumps(reference_timeline, indent=2)}

GENERATED SUMMARY UNITS:
{json.dumps(summary_for_model, indent=2)}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": SUMMARY_EVENT_MAPPING_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    result = json.loads(
        response.choices[0].message.content
    )

    return result["matches"]

In [20]:
reference_timeline = [
    {
        "reference_event_id": int(i),
        "source_note_index": int(event["source_note_index"]),
        "event": event["event"]
    }
    for i, event in enumerate(consolidated_events)
]

print("Reference events:", len(reference_timeline))

Reference events: 116


In [21]:
test_config = "Whole + Gemini"

test_matches = map_reference_events_to_summary(
    reference_timeline,
    summary_units[test_config]
)

print("Returned decisions:", len(test_matches))

matched = [
    item for item in test_matches
    if item["match"] == "MATCH"
]

print("Matched reference events:", len(matched))

print("\nFirst 10 matches:")
for item in matched[:10]:
    print(item)

Returned decisions: 116
Matched reference events: 80

First 10 matches:
{'reference_event_id': 0, 'match': 'MATCH', 'summary_unit': 1}
{'reference_event_id': 2, 'match': 'MATCH', 'summary_unit': 6}
{'reference_event_id': 3, 'match': 'MATCH', 'summary_unit': 5}
{'reference_event_id': 4, 'match': 'MATCH', 'summary_unit': 7}
{'reference_event_id': 5, 'match': 'MATCH', 'summary_unit': 9}
{'reference_event_id': 6, 'match': 'MATCH', 'summary_unit': 7}
{'reference_event_id': 7, 'match': 'MATCH', 'summary_unit': 8}
{'reference_event_id': 9, 'match': 'MATCH', 'summary_unit': 8}
{'reference_event_id': 10, 'match': 'MATCH', 'summary_unit': 8}
{'reference_event_id': 12, 'match': 'MATCH', 'summary_unit': 12}


In [22]:
temporal_mapping_results = {
    "Whole + Gemini": test_matches
}

for config, units in summary_units.items():

    # Already completed above.
    if config == "Whole + Gemini":
        continue

    print(f"Mapping: {config}")

    matches = map_reference_events_to_summary(
        reference_timeline,
        units
    )

    # Make sure GPT returned one decision for all 116 events.
    if len(matches) != len(reference_timeline):
        raise ValueError(
            f"{config}: expected {len(reference_timeline)} "
            f"decisions, got {len(matches)}"
        )

    temporal_mapping_results[config] = matches

    matched_count = sum(
        item["match"] == "MATCH"
        for item in matches
    )

    print(
        f"  Decisions: {len(matches)} | "
        f"Matched: {matched_count}"
    )

Mapping: Whole + BGE
  Decisions: 116 | Matched: 68
Mapping: Whole + MedCPT
  Decisions: 116 | Matched: 61
Mapping: Fixed + Gemini
  Decisions: 116 | Matched: 79
Mapping: Fixed + BGE
  Decisions: 116 | Matched: 61
Mapping: Fixed + MedCPT
  Decisions: 116 | Matched: 52
Mapping: Section + Gemini
  Decisions: 116 | Matched: 51
Mapping: Section + BGE
  Decisions: 116 | Matched: 67
Mapping: Section + MedCPT
  Decisions: 116 | Matched: 36


In [23]:
with open("temporal_summary_event_mappings.json", "w") as f:
    json.dump(
        temporal_mapping_results,
        f,
        indent=2
    )

print("Saved mappings:", len(temporal_mapping_results))

Saved mappings: 9


In [24]:
from itertools import combinations

import pandas as pd


def calculate_temporal_consistency(matches):
    """
    Calculate pairwise temporal-order accuracy.

    Only reference events that are actually represented in the summary
    are considered.

    Pairs mapped to the same summary unit are excluded because their
    relative order cannot be determined.
    """

    # Keep only matched reference events.
    matched_events = [
        {
            "reference_event_id": int(item["reference_event_id"]),
            "summary_unit": int(item["summary_unit"])
        }
        for item in matches
        if item["match"] == "MATCH"
        and item["summary_unit"] is not None
    ]

    # Reference event IDs already follow the authoritative
    # source chronology.
    matched_events = sorted(
        matched_events,
        key=lambda x: x["reference_event_id"]
    )

    correct_pairs = 0
    reversed_pairs = 0
    tied_pairs = 0

    for event_a, event_b in combinations(matched_events, 2):

        unit_a = event_a["summary_unit"]
        unit_b = event_b["summary_unit"]

        if unit_a < unit_b:
            correct_pairs += 1

        elif unit_a > unit_b:
            reversed_pairs += 1

        else:
            # Both events occur in the same summary sentence/unit,
            # so their internal ordering cannot be determined.
            tied_pairs += 1

    comparable_pairs = correct_pairs + reversed_pairs

    if comparable_pairs > 0:
        temporal_score = (
            correct_pairs / comparable_pairs
        ) * 100
    else:
        temporal_score = None

    return {
        "matched_events": len(matched_events),
        "correct_pairs": correct_pairs,
        "reversed_pairs": reversed_pairs,
        "tied_pairs_excluded": tied_pairs,
        "comparable_pairs": comparable_pairs,
        "temporal_consistency": temporal_score
    }

In [25]:
temporal_results = []

for config, matches in temporal_mapping_results.items():

    result = calculate_temporal_consistency(matches)

    temporal_results.append({
        "configuration": config,
        **result
    })

temporal_results_df = pd.DataFrame(temporal_results)

temporal_results_df = temporal_results_df.sort_values(
    "temporal_consistency",
    ascending=False
).reset_index(drop=True)

print(
    temporal_results_df[
        [
            "configuration",
            "matched_events",
            "correct_pairs",
            "reversed_pairs",
            "tied_pairs_excluded",
            "comparable_pairs",
            "temporal_consistency"
        ]
    ].to_string(index=False)
)

   configuration  matched_events  correct_pairs  reversed_pairs  tied_pairs_excluded  comparable_pairs  temporal_consistency
  Fixed + Gemini              79           2576             304                  201              2880             89.444444
     Whole + BGE              68           1895             309                   74              2204             85.980036
  Whole + MedCPT              61           1470             267                   93              1737             84.628670
   Section + BGE              67           1700             356                  155              2056             82.684825
     Fixed + BGE              61           1436             323                   71              1759             81.637294
Section + Gemini              51            944             257                   74              1201             78.601166
  Fixed + MedCPT              52           1002             281                   43              1283             78.098207


In [26]:
temporal_results_df.to_csv(
    "rag_temporal_consistency_results.csv",
    index=False
)

print("Saved temporal consistency results.")

Saved temporal consistency results.


**Eval 3 — Claim-level faithfulness ⏳**

In [27]:
import json

with open("rag_atomic_claims.json", "r") as f:
    claim_results = json.load(f)

print("Loaded configurations:", len(claim_results))
print(
    "Total claims:",
    sum(len(claims) for claims in claim_results.values())
)

Loaded configurations: 9
Total claims: 408


In [28]:
FAITHFULNESS_PROMPT = """
Evaluate whether one clinical-summary claim is supported by the provided
source evidence.

Judge only source-grounded CLINICAL CONTENT.

Temporal consistency is evaluated separately. Ignore temporal placement
such as "on arrival", "later", or "on discharge day" when assigning the
faithfulness label. Judge whether the underlying clinical fact is supported.

SUPPORTED:
The evidence explicitly states or sufficiently establishes the clinical claim.

UNSUPPORTED:
The clinical claim, value, diagnosis, treatment, investigation, finding,
or other substantive clinical content is not established by the evidence.

For compound claims, all substantive clinical components must be supported.

Use only the provided evidence. Do not use outside knowledge or infer
undocumented information.

Return valid JSON only:
{
  "label": "SUPPORTED" or "UNSUPPORTED",
  "reason": "brief source-based explanation",
  "evidence_quote": "shortest relevant source text, or null if unsupported"
}
"""

In [31]:
notes = pd.read_csv("../data/raw/clinical_notes.csv")

print("Original notes:", len(notes))
print(notes.columns.tolist())

Original notes: 1602
['ingest_timestamp', 'clinical_note_id', 'clean_note_text', 'creation_timestamp', 'updt_dt_tm', 'note_subject', 'note_type', 'admission_id', 'person_id']


In [32]:
# Remove the 7 records whose entire note text is "#NAME?"
notes_clean = notes[
    notes["clean_note_text"].astype(str).str.strip() != "#NAME?"
].copy()

print("After #NAME? removal:", len(notes_clean))

After #NAME? removal: 1595


In [33]:
notes_dedup = (
    notes_clean
    .sort_values(["person_id", "creation_timestamp"])
    .drop_duplicates(
        subset=["person_id", "clean_note_text"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("After deduplication:", len(notes_dedup))
print("Patients:", notes_dedup["person_id"].nunique())

After deduplication: 1103
Patients: 50


In [34]:
SELECTED_PERSON_ID = "c6c45c39-cd73-49dd-818d-0a7865fe8a7f"

reference_notes = (
    notes_dedup[
        notes_dedup["person_id"] == SELECTED_PERSON_ID
    ]
    .sort_values("creation_timestamp")
    .reset_index(drop=True)
)

print("Source notes:", len(reference_notes))
print(
    "Date range:",
    reference_notes["creation_timestamp"].min(),
    "to",
    reference_notes["creation_timestamp"].max()
)

Source notes: 45
Date range: 03/01/2026 01:35 to 10/01/2026 10:15


In [35]:
source_evidence_texts = (
    reference_notes["clean_note_text"]
    .astype(str)
    .tolist()
)

print("Source evidence notes:", len(source_evidence_texts))

Source evidence notes: 45


In [38]:
gemini_client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)


def embed_gemini_texts(texts, batch_size=50):
    """
    Embed texts using Gemini for semantic evidence retrieval.
    """

    all_embeddings = []

    for i in range(0, len(texts), batch_size):

        batch_texts = texts[i:i + batch_size]

        response = gemini_client.models.embed_content(
            model="gemini-embedding-001",
            contents=batch_texts
        )

        batch_embeddings = [
            embedding.values
            for embedding in response.embeddings
        ]

        all_embeddings.extend(batch_embeddings)

        if i + batch_size < len(texts):
            time.sleep(35)

    return np.array(all_embeddings)

In [39]:
source_gemini_embeddings = embed_gemini_texts(
    source_evidence_texts
)

print("Embedding shape:", source_gemini_embeddings.shape)

Embedding shape: (45, 3072)


In [ ]:
def retrieve_evidence_gemini(claim, top_k=5):
    """
    Retrieve the top-k most relevant source notes for one atomic claim.

    Evidence comes from the complete 45-note source-of-truth record,
    not from any RAG configuration's retrieved chunks.
    """

    # Embed the claim.
    claim_embedding = embed_gemini_texts([claim])

    # Compare the claim against all 45 source-note embeddings.
    similarities = cosine_similarity(
        claim_embedding,
        source_gemini_embeddings
    )[0]

    # Get the indices of the top-k most similar notes.
    top_indices = np.argsort(similarities)[::-1][:top_k]

    retrieved = []

    for rank, idx in enumerate(top_indices, start=1):
        retrieved.append({
            "rank": rank,
            "note_index": int(idx),
            "similarity": float(similarities[idx]),
            "evidence": source_evidence_texts[idx]
        })

    return retrieved

In [42]:
test_claim = "CTPA confirmed chronic thromboembolic disease."

test_evidence = retrieve_evidence_gemini(
    test_claim,
    top_k=5
)

for item in test_evidence:
    print(
        f"Rank {item['rank']} | "
        f"Note {item['note_index']} | "
        f"Similarity {item['similarity']:.4f}"
    )
    print(item["evidence"][:500])
    print("-" * 80)

Rank 1 | Note 6 | Similarity 0.7316
Clinician Leading Ward Round
Dr. Elizabeth Kathryn Singh (Cons)

Presenting Complaint
Progressive breathlessness hx noted thank you

Issues
1. Chronic Thromboembolic Pulmonary Hypertension (CTEPH)
-Elevated NT-proBNP (4,500 pg/mL)
-ABG: PaO2 8.5 kPa, compensated respiratory alkalosis
-Persistent tachycardia and horderline hypotension
-SpO2 90% on 2L NC

On Review
Still SOB. Feels slightly better on O2.

Investigations
CTPA and echo requested.

Test Results
Pending ABG and coag screen. ABG from E
--------------------------------------------------------------------------------
Rank 2 | Note 5 | Similarity 0.7206
Clerking Doctor
Dr. Sade Olowoyeye (SpR)

Presenting Complaint
Progressive breathlessness. Worsening SOB over wks. Now limits daily acti vity. No CP. No syncope. No palps. Worse on exertion, unchanged at rest. No fever, cough, or sputum. No recent travel or surgery.

Review of Systems
CVS: No CP, no dizziness. Res: No haemoptysis, no wheeze. GI

In [43]:
FAITHFULNESS_PROMPT = """
Evaluate whether one clinical-summary claim is supported by the provided
source evidence.

Judge only source-grounded CLINICAL CONTENT.

Temporal consistency is evaluated separately. Ignore temporal placement
such as "on arrival", "later", or "on discharge day" when assigning the
faithfulness label. Judge whether the underlying clinical fact is supported.

SUPPORTED:
The evidence explicitly states or sufficiently establishes the clinical claim.

UNSUPPORTED:
The clinical claim, value, diagnosis, treatment, investigation, finding,
or other substantive clinical content is not established by the evidence.

For compound claims, all substantive clinical components must be supported.

Use only the provided evidence. Do not use outside knowledge or infer
undocumented information.

Return valid JSON only:
{
  "label": "SUPPORTED" or "UNSUPPORTED",
  "reason": "brief source-based explanation",
  "evidence_quote": "shortest relevant source text, or null if unsupported"
}
"""

In [45]:
def verify_claim_gpt54(claim, evidence):

    # Give GPT-5.4 Mini the five retrieved source notes.
    evidence_text = "\n\n".join(
        [
            f"EVIDENCE {item['rank']}:\n{item['evidence']}"
            for item in evidence
        ]
    )

    user_message = f"""
CLAIM:
{claim}

SOURCE EVIDENCE:
{evidence_text}
"""

    response = openai_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": FAITHFULNESS_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    return json.loads(
        response.choices[0].message.content
    )

In [46]:
result = verify_claim_gpt54(
    test_claim,
    test_evidence
)

print(json.dumps(result, indent=2))

{
  "label": "SUPPORTED",
  "reason": "The source explicitly states that the CTPA confirmed chronic thromboembolic disease.",
  "evidence_quote": "CTPA confirmed chronic thromboembolic disease"
}


In [47]:
fake_claim = "The patient was diagnosed with diabetes mellitus."

fake_evidence = retrieve_evidence_gemini(
    fake_claim,
    top_k=5
)

fake_result = verify_claim_gpt54(
    fake_claim,
    fake_evidence
)

print(json.dumps(fake_result, indent=2))

{
  "label": "UNSUPPORTED",
  "reason": "The evidence lists past medical history of HTN and asthma and discusses breathlessness/pulmonary HTN, but it does not state that the patient was diagnosed with diabetes mellitus.",
  "evidence_quote": "Past Medical History\n- HTN\n- Asthma"
}


In [48]:
faithfulness_results = {}

for config_name, claims in claim_results.items():

    print(f"\nEvaluating: {config_name}")
    config_results = []

    for i, claim_item in enumerate(claims, start=1):

        claim_text = claim_item["claim"]

        # Step 1: Retrieve top-5 evidence from the complete
        # 45-note source-of-truth record.
        evidence = retrieve_evidence_gemini(
            claim_text,
            top_k=5
        )

        # Step 2: Ask GPT-5.4 Mini whether the claim
        # is supported by that evidence.
        result = verify_claim_gpt54(
            claim_text,
            evidence
        )

        # Store everything needed for later analysis/auditing.
        config_results.append({
            "claim_id": claim_item["claim_id"],
            "claim": claim_text,
            "label": result["label"],
            "reason": result.get("reason"),
            "evidence_quote": result.get("evidence_quote"),
            "retrieved_evidence": evidence
        })

        print(
            f"{i}/{len(claims)} | "
            f"{claim_item['claim_id']} | "
            f"{result['label']}"
        )

    faithfulness_results[config_name] = config_results

print("\nEvaluation complete.")


Evaluating: Whole + Gemini
1/47 | C01 | SUPPORTED
2/47 | C02 | SUPPORTED
3/47 | C03 | SUPPORTED
4/47 | C04 | SUPPORTED
5/47 | C05 | SUPPORTED
6/47 | C06 | SUPPORTED
7/47 | C07 | SUPPORTED
8/47 | C08 | SUPPORTED
9/47 | C09 | SUPPORTED
10/47 | C10 | SUPPORTED
11/47 | C11 | SUPPORTED
12/47 | C12 | SUPPORTED
13/47 | C13 | SUPPORTED
14/47 | C14 | SUPPORTED
15/47 | C15 | SUPPORTED
16/47 | C16 | SUPPORTED
17/47 | C17 | SUPPORTED
18/47 | C18 | SUPPORTED
19/47 | C19 | SUPPORTED
20/47 | C20 | SUPPORTED
21/47 | C21 | SUPPORTED
22/47 | C22 | SUPPORTED
23/47 | C23 | UNSUPPORTED
24/47 | C24 | SUPPORTED
25/47 | C25 | SUPPORTED
26/47 | C26 | UNSUPPORTED
27/47 | C27 | SUPPORTED
28/47 | C28 | UNSUPPORTED
29/47 | C29 | UNSUPPORTED
30/47 | C30 | SUPPORTED
31/47 | C31 | SUPPORTED
32/47 | C32 | SUPPORTED
33/47 | C33 | UNSUPPORTED
34/47 | C34 | UNSUPPORTED
35/47 | C35 | UNSUPPORTED
36/47 | C36 | SUPPORTED
37/47 | C37 | UNSUPPORTED
38/47 | C38 | UNSUPPORTED
39/47 | C39 | SUPPORTED
40/47 | C40 | SUPPORTED
41/

In [49]:
import json

with open("rag_faithfulness_results_gpt54mini.json", "w") as f:
    json.dump(faithfulness_results, f, indent=2)

print("Saved configurations:", len(faithfulness_results))
print(
    "Saved claims:",
    sum(len(v) for v in faithfulness_results.values())
)

Saved configurations: 9
Saved claims: 408


In [50]:
faithfulness_scores = []

for config_name, results in faithfulness_results.items():

    total = len(results)

    supported = sum(
        1 for item in results
        if item["label"] == "SUPPORTED"
    )

    unsupported = total - supported

    faithfulness = (supported / total) * 100

    faithfulness_scores.append({
        "Configuration": config_name,
        "Total Claims": total,
        "Supported": supported,
        "Unsupported": unsupported,
        "Faithfulness (%)": faithfulness
    })

faithfulness_df = pd.DataFrame(faithfulness_scores)

faithfulness_df = faithfulness_df.sort_values(
    "Faithfulness (%)",
    ascending=False
).reset_index(drop=True)

faithfulness_df.index += 1

display(faithfulness_df)

,Configuration,Total Claims,Supported,Unsupported,Faithfulness (%)
1,Section + BGE,34,28,6,82.352941
2,Whole + MedCPT,68,54,14,79.411765
3,Whole + BGE,48,37,11,77.083333
4,Whole + Gemini,47,36,11,76.595745
5,Section + Gemini,46,34,12,73.913043
6,Fixed + Gemini,52,37,15,71.153846
7,Fixed + BGE,50,34,16,68.000000
8,Section + MedCPT,28,17,11,60.714286
9,Fixed + MedCPT,35,20,15,57.142857


In [51]:
faithfulness_df.to_csv(
    "rag_faithfulness_scores_gpt54mini.csv",
    index=True
)

print("Saved.")

Saved.


In [52]:
# Build one table containing the four frozen evaluation metrics.
aggregate_df = pd.DataFrame({
    "Configuration": [
        "Whole + Gemini",
        "Whole + BGE",
        "Whole + MedCPT",
        "Fixed + Gemini",
        "Fixed + BGE",
        "Fixed + MedCPT",
        "Section + Gemini",
        "Section + BGE",
        "Section + MedCPT"
    ],

    "Coverage": [
        100.0, 100.0, 92.5,
        97.5, 100.0, 92.5,
        72.5, 92.5, 70.0
    ],

    "TFIDF": [
        0.444030,
        0.359584,
        0.431132,
        0.447463,
        0.401397,
        0.397682,
        0.503101,
        0.348051,
        0.415435
    ],

    "Faithfulness": [
        76.595745,
        77.083333,
        79.411765,
        71.153846,
        68.000000,
        57.142857,
        73.913043,
        82.352941,
        60.714286
    ],

    "Temporal": [
        76.126429,
        85.980036,
        84.628670,
        89.444444,
        81.637294,
        78.098207,
        78.601166,
        82.684825,
        64.930556
    ]
})


# Rank each configuration independently on each metric.
# Higher score = better.
# method="average" gives tied configurations the same average rank.
for metric in ["Coverage", "TFIDF", "Faithfulness", "Temporal"]:
    aggregate_df[f"{metric}_Rank"] = (
        aggregate_df[metric]
        .rank(ascending=False, method="average")
    )


# Our predefined aggregate rule:
# add the four ranks; LOWER rank sum = better overall performance.
rank_columns = [
    "Coverage_Rank",
    "TFIDF_Rank",
    "Faithfulness_Rank",
    "Temporal_Rank"
]

aggregate_df["Rank_Sum"] = aggregate_df[rank_columns].sum(axis=1)


# Final overall ranking.
aggregate_df = (
    aggregate_df
    .sort_values("Rank_Sum")
    .reset_index(drop=True)
)

aggregate_df["Overall_Rank"] = range(1, len(aggregate_df) + 1)


display(
    aggregate_df[
        [
            "Overall_Rank",
            "Configuration",
            "Coverage_Rank",
            "TFIDF_Rank",
            "Faithfulness_Rank",
            "Temporal_Rank",
            "Rank_Sum"
        ]
    ]
)

,Overall_Rank,Configuration,Coverage_Rank,TFIDF_Rank,Faithfulness_Rank,Temporal_Rank,Rank_Sum
0,1,Fixed + Gemini,4.0,2.0,6.0,1.0,13.0
1,2,Whole + BGE,2.0,8.0,3.0,2.0,15.0
2,3,Whole + MedCPT,6.0,4.0,2.0,3.0,15.0
3,4,Whole + Gemini,2.0,3.0,4.0,8.0,17.0
4,5,Fixed + BGE,2.0,6.0,7.0,5.0,20.0
5,6,Section + Gemini,8.0,1.0,5.0,6.0,20.0
6,7,Section + BGE,6.0,9.0,1.0,4.0,20.0
7,8,Fixed + MedCPT,6.0,7.0,9.0,7.0,29.0
8,9,Section + MedCPT,9.0,5.0,8.0,9.0,31.0


In [53]:
aggregate_df.to_csv(
    "rag_configuration_aggregate_ranking.csv",
    index=False
)

print("Saved.")

Saved.


In [54]:
# Sensitivity analysis:
# Recalculate aggregate ranking WITHOUT TF-IDF.
#
# This does NOT replace our primary four-metric selection rule.
# It simply tests how dependent the selected configuration is
# on inclusion of the lexical-similarity metric.

aggregate_df["Rank_Sum_No_TFIDF"] = (
    aggregate_df["Coverage_Rank"]
    + aggregate_df["Faithfulness_Rank"]
    + aggregate_df["Temporal_Rank"]
)

sensitivity_df = (
    aggregate_df[
        [
            "Configuration",
            "Coverage_Rank",
            "Faithfulness_Rank",
            "Temporal_Rank",
            "Rank_Sum_No_TFIDF"
        ]
    ]
    .sort_values("Rank_Sum_No_TFIDF")
    .reset_index(drop=True)
)

sensitivity_df["Sensitivity_Rank"] = range(
    1,
    len(sensitivity_df) + 1
)

display(sensitivity_df)

,Configuration,Coverage_Rank,Faithfulness_Rank,Temporal_Rank,Rank_Sum_No_TFIDF,Sensitivity_Rank
0,Whole + BGE,2.0,3.0,2.0,7.0,1
1,Fixed + Gemini,4.0,6.0,1.0,11.0,2
2,Whole + MedCPT,6.0,2.0,3.0,11.0,3
3,Section + BGE,6.0,1.0,4.0,11.0,4
4,Whole + Gemini,2.0,4.0,8.0,14.0,5
5,Fixed + BGE,2.0,7.0,5.0,14.0,6
6,Section + Gemini,8.0,5.0,6.0,19.0,7
7,Fixed + MedCPT,6.0,9.0,7.0,22.0,8
8,Section + MedCPT,9.0,8.0,9.0,26.0,9
